# Ablation study: what each design choice is worth

For each subset, one design choice is removed or changed at a time and scored with the **same 5-fold cross-validation on the non-validation training engines** (paired folds; no test data). The base model is HistGradientBoosting for every variant, so only the design choice changes; model choice is a separate block.

| Block | Variants |
|---|---|
| Features | raw sensors only · no trend (slope) · no drift-from-own-baseline · no cycle count |
| Windows | shorter (5/10/5) · longer (30/50/30) vs current (10/20/10) |
| Target | RUL cap 110 · 140 · none vs 125 |
| Regimes | no per-condition normalisation (FD002 / FD004 only) |
| Model | Ridge · XGBoost · HistGB + XGBoost average |
| Calibration | raw quantile coverage vs conformal-calibrated (cross-fitted), overall and for RUL ≤ 30 |

**Metrics** (held-out "test-like" rows): `rmse_capped` (vs min(true RUL, 125), the same truth for every variant), `rmse_true`, `rmse_true_rul_le_60` (where decisions happen), `nasa_capped_mean`, the spread across folds, and the paired difference vs baseline (`delta_vs_baseline`, negative = better; `folds_better_than_baseline`).

**Run on Kaggle** like the main notebook (dataset attached, GPU, Internet not needed). Roughly 10–20 minutes per subset. Output: `/kaggle/working/ablation_outputs.zip`. Generated from `scripts/build_notebook.py`; edit it there.

In [ ]:
import json

# pipeline code cells + ablation code, generated from scripts/build_notebook.py
ABLATION_CELLS = json.loads(r'''[
"import json, shutil, subprocess, time, warnings\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, IsolationForest\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nwarnings.filterwarnings(\"ignore\", category=UserWarning)\nwarnings.filterwarnings(\"ignore\", category=FutureWarning)\ntry:\n    import xgboost as xgb\nexcept ImportError:\n    xgb = None\n\nSUBSET = \"FD001\"  # FD001 / FD002 / FD003 / FD004 (run_all_subsets.ipynb rewrites this line)\nSEED = 42\nRUN_TAG = \"\"  # run_all_subsets sets e.g. \"_seed1\" for extra seeds, so their outputs don't overwrite seed 42's\n\nSUBSET_CONFIG = {\n    \"FD001\": {\"condition_norm\": False, \"fault_modes\": 1},\n    \"FD002\": {\"condition_norm\": True, \"fault_modes\": 1},\n    \"FD003\": {\"condition_norm\": False, \"fault_modes\": 2},\n    \"FD004\": {\"condition_norm\": True, \"fault_modes\": 2},\n}\n\nCFG = {\n    \"subset\": SUBSET, **SUBSET_CONFIG[SUBSET],\n    # labels\n    \"rul_cap\": 125,             # piecewise-linear target: flat while the engine is healthy\n    \"fail_window\": 30,          # \"failing\" class for the early-warning classifier: RUL <= 30\n    # cleaning / features\n    \"min_unique\": 2,            # drop sensors with <= 2 distinct values within a condition\n    \"clip_quantiles\": (0.001, 0.999),\n    # data-quality gate (all fitted on training engines)\n    \"op_margin_frac\": 0.5,        # single-condition subsets: settings beyond the training range by this much = unseen\n    \"sensor_range_margin\": 0.5,   # reading beyond the training min/max by 50% of the range = out of range\n    \"sensor_range_persist\": 2,    # ... for at least this many consecutive cycles\n    \"sensor_stuck_cycles\": 15,    # identical reading this long on a sensor that never flatlines in training = stuck\n    \"roll_window\": 10, \"slope_window\": 20, \"ewm_span\": 10, \"baseline_cycles\": 10,\n    # splits (by engine)\n    \"val_frac\": 0.20, \"new_frac\": 0.30,\n    \"cv_folds\": 5,\n    # 'test-like' evaluation rows: mirror how the official test engines are cut.\n    # Test RUL reaches ~195 in FD002/FD004, so the window covers RUL up to 200.\n    \"eval_min_cycles\": 31, \"eval_max_rul\": 200,\n    # anomaly detector: trained on early-life cycles (no failure labels needed; a real fleet would use the\n    # first cycles after commissioning or maintenance). Starts after the rolling window is full.\n    \"anomaly_train_cycles\": (11, 40), \"anomaly_quantile\": 0.99, \"anomaly_persist\": 3,\n    \"anomaly_healthy_rul\": 125,  # evaluation only: false-alarm rate is measured on RUL > 125 (a proxy)\n    # RUL interval: conformal calibration of the 10% / 90% quantile models\n    \"interval_coverage\": 0.80, \"review_width_quantile\": 0.90,\n    # retraining / promotion\n    \"retrain_min_new_frac\": 0.20, \"psi_alert\": 0.20, \"promote_max_nasa_ratio\": 1.10,\n    \"promote_min_prob\": 0.80,     # how sure the bootstrap must be (80% of resamples of the validation engines)\n    # retraining here is always \"same design, more data\", so v2 is promoted unless it is clearly WORSE\n    # (non-inferiority); \"superiority\" (v2 must be clearly better) is the stricter test for design changes\n    \"promotion_test\": \"non_inferiority\",\n    \"noninferiority_margin\": 0.5,  # cycles of capped RMSE: \"clearly worse\" means worse by more than this (< fold spread)\n    # after promotion, retrain the winning design on every training engine and deploy that\n    \"full_refit\": False,\n}\n\nON_KAGGLE = Path(\"/kaggle/input\").exists()\nWORK = Path(\"/kaggle/working\") if ON_KAGGLE else Path.cwd() / \"outputs\"\nOUT = WORK / \"artifacts\" / f\"{SUBSET}{RUN_TAG}\"\n(OUT / \"plots\").mkdir(parents=True, exist_ok=True)\n(OUT / \"models\").mkdir(parents=True, exist_ok=True)\nnp.random.seed(SEED)\n\n\ndef gpu_available():\n    try:\n        r = subprocess.run([\"nvidia-smi\", \"-L\"], capture_output=True, text=True)\n        return r.returncode == 0 and \"GPU\" in r.stdout\n    except FileNotFoundError:\n        return False\n\n\ndef now_iso():\n    return datetime.now(timezone.utc).isoformat(timespec=\"seconds\")\n\n\ndef json_default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, (Path,)):\n        return str(o)\n    raise TypeError(type(o))\n\n\ndef save_json(obj, path):\n    Path(path).write_text(json.dumps(obj, indent=2, default=json_default))\n\n\nGPU = gpu_available()\nprint(f\"subset={SUBSET}  gpu={GPU}  xgboost={xgb.__version__ if xgb else 'not installed'}  outputs={OUT}\")",
"COLS = [\"unit\", \"cycle\", \"op1\", \"op2\", \"op3\"] + [f\"s{i}\" for i in range(1, 22)]\nOPS, SENSORS = COLS[2:5], COLS[5:]\n\n\ndef find_data_dir(subset):\n    roots = [Path(\"/kaggle/input\"), Path.cwd() / \"data\" / \"csv\", Path.cwd().parent / \"data\" / \"csv\",\n             Path.cwd() / \"DataSet\", Path.cwd().parent / \"DataSet\"]\n    for root in roots:\n        if root.exists():\n            for ext in (\"csv\", \"txt\"):\n                hits = sorted(root.rglob(f\"train_{subset}.{ext}\"))\n                if hits:\n                    return hits[0].parent, ext\n    raise FileNotFoundError(f\"train_{subset}.csv / .txt not found under {[str(r) for r in roots]}\")\n\n\ndef read_table(path, ext):\n    if ext == \"csv\":\n        return pd.read_csv(path)\n    df = pd.read_csv(path, sep=r\"\\s+\", header=None)\n    return df.iloc[:, :len(COLS)].set_axis(COLS, axis=1)\n\n\ndef validate_raw(df, name):\n    missing = [c for c in COLS if c not in df.columns]\n    if missing:\n        raise ValueError(f\"{name}: missing columns {missing}\")\n    num = df[COLS].apply(pd.to_numeric, errors=\"coerce\")\n    report = {\n        \"rows\": len(df),\n        \"engines\": int(num.unit.nunique()),\n        \"missing_cells\": int(df[COLS].isna().sum().sum()),\n        \"non_numeric_cells\": int(num.isna().sum().sum() - df[COLS].isna().sum().sum()),\n        \"duplicate_unit_cycle\": int(num.duplicated(subset=[\"unit\", \"cycle\"]).sum()),\n        \"cycle_gaps\": int(num.sort_values([\"unit\", \"cycle\"]).groupby(\"unit\").cycle.diff().dropna().ne(1).sum()),\n    }\n    if report[\"rows\"] == 0:\n        raise ValueError(f\"{name}: empty file\")\n    return report\n\n\ndef load_subset(subset):\n    d, ext = find_data_dir(subset)\n    train = read_table(d / f\"train_{subset}.{ext}\", ext)\n    test = read_table(d / f\"test_{subset}.{ext}\", ext)\n    if ext == \"csv\":\n        r = pd.read_csv(d / f\"RUL_{subset}.csv\")\n        final = (r.sort_values(\"unit\") if \"unit\" in r else r)[\"rul\"].to_numpy()\n    else:\n        final = pd.read_csv(d / f\"RUL_{subset}.txt\", sep=r\"\\s+\", header=None)[0].to_numpy()\n    reports = {\"train\": validate_raw(train, \"train\"), \"test\": validate_raw(test, \"test\")}\n    test_units = np.sort(pd.to_numeric(test.unit).unique())\n    if len(final) != len(test_units):\n        raise ValueError(f\"RUL file has {len(final)} values but test has {len(test_units)} engines\")\n    rul_final = pd.DataFrame({\"unit\": test_units.astype(int), \"rul_final\": final.astype(int)})\n    return train[COLS], test[COLS], rul_final, reports, d\n\n\nraw_train, raw_test, rul_final, ingest_report, DATA_DIR = load_subset(SUBSET)\nprint(\"data dir:\", DATA_DIR)\ndisplay(pd.DataFrame(ingest_report).T)",
"def basic_clean(df):\n    df = df[COLS].apply(pd.to_numeric, errors=\"coerce\").dropna(subset=[\"unit\", \"cycle\"])\n    df[[\"unit\", \"cycle\"]] = df[[\"unit\", \"cycle\"]].astype(int)\n    df = df.drop_duplicates(subset=[\"unit\", \"cycle\"], keep=\"last\").sort_values([\"unit\", \"cycle\"]).reset_index(drop=True)\n    vals = OPS + SENSORS\n    df[\"imputed\"] = df[vals].isna().any(axis=1)\n    # past-only: forward fill within each engine; leading gaps become the training mean after normalisation\n    df[vals] = df.groupby(\"unit\")[vals].ffill()\n    return df\n\n\ndef add_labels(df, cfg, rul_final=None):\n    df = df.copy()\n    last = df.groupby(\"unit\").cycle.transform(\"max\")\n    end = 0 if rul_final is None else df.unit.map(rul_final.set_index(\"unit\").rul_final)\n    df[\"rul\"] = (end + last - df.cycle).astype(int)\n    df[\"rul_capped\"] = df.rul.clip(upper=cfg[\"rul_cap\"])\n    df[\"fail\"] = (df.rul <= cfg[\"fail_window\"]).astype(int)\n    return df\n\n\ntrain_df = add_labels(basic_clean(raw_train), CFG)\ntest_df = add_labels(basic_clean(raw_test), CFG, rul_final)\n\ncheck = test_df.groupby(\"unit\").rul.min().reset_index().merge(rul_final, on=\"unit\")\nassert (check.rul == check.rul_final).all(), \"test labels do not match the RUL file\"\nprint(f\"train: {train_df.unit.nunique()} engines, {len(train_df):,} rows | test: {test_df.unit.nunique()} engines, {len(test_df):,} rows\")\nprint(f\"imputed rows: train {int(train_df.imputed.sum())}, test {int(test_df.imputed.sum())}\")\nprint(f\"class balance (fail = RUL <= {CFG['fail_window']}): {train_df.fail.mean():.1%} positive\")",
"META = [\"unit\", \"cycle\", \"condition\", \"imputed\", \"unseen_condition\", \"sensor_fault\", \"sensor_fault_detail\",\n        \"rul\", \"rul_capped\", \"fail\"]\n\n\ndef condition_key(df):\n    return (df[OPS].round(0) + 0.0).astype(str).agg(\"/\".join, axis=1)\n\n\ndef assign_condition(df, prep):\n    # operating condition per row; settings never seen in training fall back to the nearest known condition and\n    # are flagged, instead of silently using global statistics\n    if not prep[\"condition_norm\"]:\n        lo = pd.Series({o: prep[\"op_range\"][o][0] for o in OPS})\n        hi = pd.Series({o: prep[\"op_range\"][o][1] for o in OPS})\n        margin = (hi - lo) * prep[\"quality\"][\"op_margin_frac\"] + 0.01 * np.maximum(1.0, np.maximum(lo.abs(), hi.abs()))\n        unseen = ((df[OPS] < lo - margin) | (df[OPS] > hi + margin)).any(axis=1)\n        return pd.Series(\"all\", index=df.index), unseen\n    key = condition_key(df)\n    unseen = ~key.isin(list(prep[\"stats\"]))\n    if unseen.any():\n        names = list(prep[\"centers\"])\n        centers = np.array([prep[\"centers\"][n] for n in names], dtype=float)\n        scale = np.where(centers.std(axis=0) > 0, centers.std(axis=0), 1.0)\n        x = df.loc[unseen, OPS].to_numpy(float)\n        dist = np.nansum(((x[:, None, :] - centers[None]) / scale) ** 2, axis=2)\n        nearest = np.array(names, dtype=object)[dist.argmin(axis=1)]\n        nearest[np.isnan(x).all(axis=1)] = prep[\"default_condition\"]\n        key = key.copy()\n        key.loc[unseen] = nearest\n    return key, unseen\n\n\ndef _flat(frame, units, window):\n    # identical for the whole window: max - min is exactly 0 (a rolling std leaves rounding residue up to ~1e-5)\n    return (_rolling_by_unit(frame, units, window, \"max\") - _rolling_by_unit(frame, units, window, \"min\")) == 0\n\n\ndef _rolling_by_unit(frame, units, window, how):\n    # per-engine trailing window statistic, aligned back to the frame's row order\n    r = frame.groupby(units.to_numpy()).rolling(window, min_periods=window)\n    return getattr(r, how)().reset_index(level=0, drop=True).reindex(frame.index)\n\n\ndef sensor_quality(df, prep):\n    # past-only sensor-health checks: dead (never reported), stuck (flat for K cycles on a sensor that never\n    # flatlines in training) and out of the physically seen range for >= persist cycles\n    q, sensors = prep[\"quality\"], prep[\"sensors\"]\n    raw = df[sensors].astype(float)\n    lo = pd.DataFrame(prep[\"sensor_range\"][\"lo\"]).T.loc[df[\"condition\"], sensors].to_numpy()\n    hi = pd.DataFrame(prep[\"sensor_range\"][\"hi\"]).T.loc[df[\"condition\"], sensors].to_numpy()\n    span, x = hi - lo, raw.to_numpy()\n    oor = (x < lo - q[\"range_margin\"] * span) | (x > hi + q[\"range_margin\"] * span)\n    # an unseen operating condition shifts every reading; that is flagged separately, not blamed on the sensors\n    oor &= ~df[\"unseen_condition\"].to_numpy()[:, None]\n    oor = pd.DataFrame(oor, index=df.index, columns=sensors).astype(float)\n    oor = _rolling_by_unit(oor, df[\"unit\"], q[\"range_persist\"], \"min\").fillna(0).astype(bool)\n    stuck = pd.DataFrame(False, index=df.index, columns=sensors)\n    if prep[\"stuck_sensors\"]:\n        stuck[prep[\"stuck_sensors\"]] = _flat(raw[prep[\"stuck_sensors\"]], df[\"unit\"], q[\"stuck_cycles\"])\n    dead = raw.isna()\n    fault = (oor | stuck | dead).any(axis=1)\n    detail = pd.Series(\"\", index=df.index)\n    for i in np.flatnonzero(fault.to_numpy()):\n        row = df.index[i]\n        parts = [f\"{s} dead\" for s in sensors if dead.at[row, s]] + \\\n                [f\"{s} stuck\" for s in sensors if stuck.at[row, s]] + \\\n                [f\"{s} out of range\" for s in sensors if oor.at[row, s]]\n        detail.at[row] = \"; \".join(parts)\n    return fault, detail\n\n\ndef _normalise(df, prep, sensors):\n    out = pd.DataFrame(index=df.index, columns=sensors, dtype=float)\n    for c, g in df.groupby(\"condition\"):\n        st_ = prep[\"stats\"].get(c, prep[\"global\"])\n        mu = pd.Series(st_[\"mean\"])[sensors]\n        sd = pd.Series(st_[\"std\"])[sensors].replace(0, np.nan)\n        out.loc[g.index] = ((g[sensors] - mu) / sd).to_numpy()\n    return out.fillna(0.0)\n\n\ndef fit_prep(df, cfg):\n    df = df.assign(condition=condition_key(df) if cfg[\"condition_norm\"] else \"all\")\n    stats = {c: {\"mean\": g[SENSORS].mean().to_dict(), \"std\": g[SENSORS].std().to_dict()}\n             for c, g in df.groupby(\"condition\")}\n    glob = {\"mean\": df[SENSORS].mean().to_dict(), \"std\": df[SENSORS].std().to_dict()}\n    nunique = df.groupby(\"condition\")[SENSORS].nunique().median()\n    sensors = [s for s in SENSORS if nunique[s] > cfg[\"min_unique\"]]\n    prep = {\"sensors\": sensors, \"stats\": stats, \"global\": glob, \"condition_norm\": cfg[\"condition_norm\"]}\n    z = _normalise(df, prep, sensors)\n    lo, hi = cfg[\"clip_quantiles\"]\n    prep[\"clip\"] = {s: (float(z[s].quantile(lo)), float(z[s].quantile(hi))) for s in sensors}\n    prep[\"dropped\"] = [s for s in SENSORS if s not in sensors]\n    # what \"normal\" looks like, for the unseen-condition check and the sensor-health gate\n    grouped = df.groupby(\"condition\")\n    prep[\"centers\"] = {c: [float(v) for v in g[OPS].mean()] for c, g in grouped}\n    prep[\"default_condition\"] = str(df[\"condition\"].value_counts().idxmax())\n    prep[\"op_range\"] = {o: [float(df[o].min()), float(df[o].max())] for o in OPS}\n    prep[\"sensor_range\"] = {\"lo\": grouped[sensors].min().T.to_dict(), \"hi\": grouped[sensors].max().T.to_dict()}\n    k = cfg[\"sensor_stuck_cycles\"]\n    flat = _flat(df[sensors].astype(float), df[\"unit\"], k)\n    prep[\"stuck_sensors\"] = [s for s in sensors if flat[s].mean() < 1e-4]  # never flat in training -> checkable\n    prep[\"quality\"] = {\"op_margin_frac\": cfg[\"op_margin_frac\"], \"range_margin\": cfg[\"sensor_range_margin\"],\n                       \"range_persist\": cfg[\"sensor_range_persist\"], \"stuck_cycles\": k}\n    return prep\n\n\ndef apply_prep(df, prep):\n    df = df.sort_values([\"unit\", \"cycle\"]).reset_index(drop=True)\n    condition, unseen = assign_condition(df, prep)\n    df = df.assign(condition=condition, unseen_condition=unseen.to_numpy())\n    fault, detail = sensor_quality(df, prep)\n    df = df.assign(sensor_fault=fault.to_numpy(), sensor_fault_detail=detail.to_numpy())\n    sensors = prep[\"sensors\"]\n    z = _normalise(df, prep, sensors)\n    for s in sensors:\n        z[s] = z[s].clip(*prep[\"clip\"][s])\n    return pd.concat([df[[c for c in META if c in df]], z], axis=1)\n\n\ndef _rolling_sum(a, w):\n    c = np.cumsum(a, axis=0)\n    out = c.copy()\n    out[w:] = c[w:] - c[:-w]\n    return out\n\n\ndef _rolling_slope(y, w):\n    # least-squares slope over the last w cycles (fewer at the start of an engine's life)\n    n_rows = len(y)\n    t = np.arange(n_rows, dtype=float)[:, None]\n    n = np.minimum(np.arange(1, n_rows + 1), w)[:, None].astype(float)\n    st, stt = _rolling_sum(t, w), _rolling_sum(t * t, w)\n    sy, sty = _rolling_sum(y, w), _rolling_sum(t * y, w)\n    den = n * stt - st ** 2\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        slope = (n * sty - st * sy) / den\n    return np.where(den > 1e-9, slope, 0.0)\n\n\ndef feature_cols(prep, cfg):\n    w, sw, span = cfg[\"roll_window\"], cfg[\"slope_window\"], cfg[\"ewm_span\"]\n    sufs = [\"\", f\"_mean{w}\", f\"_std{w}\", f\"_slope{sw}\", f\"_ewm{span}\", \"_drift\"]\n    return [\"cycle\"] + [f\"{s}{suf}\" for suf in sufs for s in prep[\"sensors\"]]\n\n\ndef anomaly_cols(prep, cfg):\n    return [f\"{s}_mean{cfg['roll_window']}\" for s in prep[\"sensors\"]]\n\n\ndef build_features(dz, sensors, cfg):\n    w, sw, span, base_n = cfg[\"roll_window\"], cfg[\"slope_window\"], cfg[\"ewm_span\"], cfg[\"baseline_cycles\"]\n    blocks = []\n    for _, g in dz.groupby(\"unit\", sort=True):\n        y = g[sensors].to_numpy(float)\n        frame = pd.DataFrame(y)\n        roll = frame.rolling(w, min_periods=1)\n        ewm = frame.ewm(span=span, adjust=False).mean().to_numpy()\n        base = np.cumsum(y, axis=0) / np.arange(1, len(y) + 1)[:, None]\n        if len(y) > base_n:\n            base[base_n:] = base[base_n - 1]\n        blocks.append(np.hstack([y, roll.mean().to_numpy(), roll.std().fillna(0.0).to_numpy(),\n                                 _rolling_slope(y, sw), ewm, ewm - base]))\n    names = feature_cols({\"sensors\": sensors}, cfg)[1:]\n    X = pd.DataFrame(np.vstack(blocks), columns=names, index=dz.index)\n    return pd.concat([dz[[c for c in META if c in dz]], X], axis=1)\n\n\ndef featurize(df, prep, cfg):\n    return build_features(apply_prep(df, prep), prep[\"sensors\"], cfg)\n\n\ndef check_causality(df, prep, cfg, n_units=8, seed=SEED):\n    rng = np.random.default_rng(seed)\n    full = featurize(df, prep, cfg).set_index([\"unit\", \"cycle\"])\n    feats = feature_cols(prep, cfg)[1:]\n    for u in rng.choice(df.unit.unique(), n_units, replace=False):\n        g = df[df.unit == u]\n        cut = int(rng.integers(1, len(g) + 1))\n        part = featurize(g.iloc[:cut], prep, cfg).iloc[-1]\n        ref = full.loc[(u, int(part.cycle)), feats].to_numpy(float)\n        assert np.allclose(part[feats].to_numpy(float), ref, atol=1e-8), f\"feature leak at engine {u}\"\n    return True\n\n\ndef check_cleaning_causality(raw, prep, cfg, n_units=8, seed=SEED):\n    # inject gaps (dense at the start of the series), then check cleaning + features stay past-only\n    rng = np.random.default_rng(seed)\n    feats = feature_cols(prep, cfg)[1:]\n    for u in rng.choice(raw.unit.unique(), n_units, replace=False):\n        g = raw[raw.unit == u].copy()\n        gaps = rng.random((len(g), len(SENSORS))) < 0.05\n        gaps[:3] |= rng.random((3, len(SENSORS))) < 0.5\n        g[SENSORS] = g[SENSORS].mask(gaps)\n        full = featurize(basic_clean(g), prep, cfg).set_index(\"cycle\")\n        cut = int(rng.integers(1, len(g) + 1))\n        part = featurize(basic_clean(g.iloc[:cut]), prep, cfg).iloc[-1]\n        ref = full.loc[int(part.cycle), feats].to_numpy(float)\n        assert np.allclose(part[feats].to_numpy(float), ref, atol=1e-8), f\"cleaning leak at engine {u}\"\n    return True\n\n\n_prep_demo = fit_prep(train_df, CFG)\n_feat_demo = featurize(train_df, _prep_demo, CFG)\nprint(\"sensors kept   :\", _prep_demo[\"sensors\"])\nprint(\"sensors dropped:\", _prep_demo[\"dropped\"])\nprint(\"conditions     :\", len(_prep_demo[\"stats\"]))\nprint(f\"features       : {len(feature_cols(_prep_demo, CFG))} per row, matrix {_feat_demo.shape}\")\nassert not _feat_demo[feature_cols(_prep_demo, CFG)].isna().any().any()\nprint(\"causality check:\", check_causality(train_df, _prep_demo, CFG))\nprint(\"cleaning causality check (injected gaps):\", check_cleaning_causality(raw_train, _prep_demo, CFG))",
"def split_units(df, cfg, seed=SEED):\n    units = np.array(sorted(df.unit.unique()))\n    np.random.default_rng(seed).shuffle(units)\n    n_val, n_new = round(len(units) * cfg[\"val_frac\"]), round(len(units) * cfg[\"new_frac\"])\n    return {\"val\": sorted(units[:n_val].tolist()),\n            \"new\": sorted(units[n_val:n_val + n_new].tolist()),\n            \"initial\": sorted(units[n_val + n_new:].tolist())}\n\n\nSPLITS = split_units(train_df, CFG)\nassert not (set(SPLITS[\"val\"]) & set(SPLITS[\"new\"]) | set(SPLITS[\"val\"]) & set(SPLITS[\"initial\"])\n            | set(SPLITS[\"new\"]) & set(SPLITS[\"initial\"])), \"engine leak between splits\"\nval_df = train_df[train_df.unit.isin(SPLITS[\"val\"])].reset_index(drop=True)\nprint({k: len(v) for k, v in SPLITS.items()}, \"engines\")",
"def make_rul_model(kind):\n    if kind == \"ridge\":\n        return make_pipeline(StandardScaler(), Ridge(alpha=1.0))\n    if kind == \"hgb\":\n        return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,\n                                             l2_regularization=1.0, early_stopping=False, random_state=SEED)\n    if kind == \"xgb\":\n        return xgb.XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=5, subsample=0.8,\n                                colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, tree_method=\"hist\",\n                                device=\"cuda\" if GPU else \"cpu\", random_state=SEED, n_jobs=-1)\n    raise ValueError(kind)\n\n\ndef make_quantile(q):\n    return HistGradientBoostingRegressor(loss=\"quantile\", quantile=q, max_iter=300, learning_rate=0.05,\n                                         max_leaf_nodes=31, min_samples_leaf=20, early_stopping=False,\n                                         random_state=SEED)\n\n\ndef make_classifier():\n    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,\n                                          class_weight=\"balanced\", early_stopping=False, random_state=SEED)\n\n\nCANDIDATES = [\"ridge\", \"hgb\"] + ([\"xgb\"] if xgb is not None else [])\n\n\ndef nasa_score(y_true, y_pred):\n    d = np.asarray(y_pred, float) - np.asarray(y_true, float)\n    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)\n\n\ndef rul_metrics(y_true, y_pred, cap):\n    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)\n    d = y_pred - y_true\n    s = nasa_score(y_true, y_pred)\n    s_cap = nasa_score(np.minimum(y_true, cap), y_pred)\n    return {\"rmse\": float(np.sqrt(np.mean(d ** 2))),\n            \"rmse_capped\": float(np.sqrt(np.mean((y_pred - np.minimum(y_true, cap)) ** 2))),\n            \"mae\": float(np.mean(np.abs(d))), \"bias\": float(np.mean(d)),\n            \"late_pct\": float(np.mean(d > 0) * 100),\n            \"nasa_score\": float(s.sum()), \"nasa_mean\": float(s.mean()),\n            \"nasa_capped_mean\": float(s_cap.mean()), \"n\": int(len(d))}\n\n\ndef clf_metrics(y, p, threshold):\n    y, p = np.asarray(y), np.asarray(p)\n    pred = p >= threshold\n    tp, fp, fn = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum()), int((~pred & (y == 1)).sum())\n    prec = tp / (tp + fp) if tp + fp else 0.0\n    rec = tp / (tp + fn) if tp + fn else 0.0\n    two_classes = len(np.unique(y)) == 2\n    return {\"pr_auc\": float(average_precision_score(y, p)) if two_classes else float(\"nan\"),\n            \"roc_auc\": float(roc_auc_score(y, p)) if two_classes else float(\"nan\"),\n            \"precision\": prec, \"recall\": rec,\n            \"f2\": 5 * prec * rec / (4 * prec + rec) if prec + rec else 0.0,\n            \"positive_rate\": float(y.mean())}\n\n\ndef anomaly_metrics(P, cfg):\n    healthy = P.rul > cfg[\"anomaly_healthy_rul\"]\n    near = P.rul <= cfg[\"fail_window\"]\n    alarms = P[P.anomaly_persistent & ~healthy]\n    first = alarms.groupby(\"unit\").rul.max()\n    return {\"false_alarm_rate_healthy\": float(P.loc[healthy, \"anomaly_persistent\"].mean()) if healthy.any() else float(\"nan\"),\n            \"detection_rate_last30\": float(P.loc[near, \"anomaly_persistent\"].mean()) if near.any() else float(\"nan\"),\n            \"engines_detected_pct\": float(len(first) / P.unit.nunique() * 100),\n            \"median_rul_at_first_alarm\": float(first.median()) if len(first) else float(\"nan\")}\n\n\ndef interval_coverage(P):\n    # quantile models are trained on the capped target, so coverage is measured against it\n    return float(((P.rul_capped >= P.rul_lo) & (P.rul_capped <= P.rul_hi)).mean())\n\n\ndef coverage_by_band(P, name, bands=((0, 30), (30, 60), (60, 125))):\n    # coverage where decisions are made; CRITICAL relies on the lower bound when RUL <= 30\n    out = {}\n    for lo, hi in bands:\n        b = P[(P.rul > lo) & (P.rul <= hi)]\n        if len(b):\n            out[f\"{name}_coverage_rul_{lo}_{hi}\"] = interval_coverage(b)\n            out[f\"{name}_below_lower_bound_rul_{lo}_{hi}\"] = float((b.rul_capped < b.rul_lo).mean())\n    return out\n\n\ndef bootstrap_gain(P_old, P_new, cfg, n_boot=2000, seed=SEED):\n    # resample validation engines: how often is the new model better on capped RMSE, and how often is its\n    # capped NASA score more than the allowed ratio worse?\n    cap = cfg[\"rul_cap\"]\n    def per_engine(P):\n        P = test_like(P, cfg)\n        y = np.minimum(P.rul, cap)\n        return P.assign(se=(P.rul_pred - y) ** 2, nasa=nasa_score(y, P.rul_pred)).groupby(\"unit\")[[\"se\", \"nasa\"]] \\\n                .agg([\"sum\", \"count\"])\n    a = per_engine(P_old)\n    b = per_engine(P_new).reindex(a.index)\n    pick = np.random.default_rng(seed).integers(0, len(a), size=(n_boot, len(a)))\n    mean = lambda s, col: s[(col, \"sum\")].to_numpy()[pick].sum(axis=1) / s[(col, \"count\")].to_numpy()[pick].sum(axis=1)\n    diff = np.sqrt(mean(b, \"se\")) - np.sqrt(mean(a, \"se\"))\n    nasa_ratio = mean(b, \"nasa\") / np.maximum(mean(a, \"nasa\"), 1e-12)\n    return {\"prob_new_better\": float((diff < 0).mean()), \"prob_new_worse\": float((diff > 0).mean()),\n            \"prob_new_worse_by_margin\": float((diff > cfg[\"noninferiority_margin\"]).mean()),\n            \"rmse_diff_ci90\": [float(np.quantile(diff, 0.05)), float(np.quantile(diff, 0.95))],\n            \"prob_nasa_clearly_worse\": float((nasa_ratio > cfg[\"promote_max_nasa_ratio\"]).mean()),\n            \"nasa_ratio_ci90\": [float(np.quantile(nasa_ratio, 0.05)), float(np.quantile(nasa_ratio, 0.95))]}\n\n\ndef conformal_correction(y, lo, hi, coverage):\n    # conformalized quantile regression: how far outside [lo, hi] the truth falls, at the target coverage\n    scores = np.maximum(np.asarray(lo) - np.asarray(y), np.asarray(y) - np.asarray(hi))\n    level = min(1.0, np.ceil((len(scores) + 1) * coverage) / len(scores))\n    return float(np.quantile(scores, level))\n\n\ndef prefix(p, d):\n    return {f\"{p}_{k}\": v for k, v in d.items()}\n\n\ndef latest(P):\n    return P.sort_values([\"unit\", \"cycle\"]).groupby(\"unit\").tail(1).reset_index(drop=True)\n\n\ndef test_like(P, cfg):\n    return P[(P.cycle >= cfg[\"eval_min_cycles\"]) & (P.rul <= cfg[\"eval_max_rul\"])]\n\n\ndef train_bundle(df, kind, cfg, clf_threshold, interval_correction=0.0):\n    t0 = time.time()\n    prep = fit_prep(df, cfg)\n    F = featurize(df, prep, cfg)\n    feats, acols = feature_cols(prep, cfg), anomaly_cols(prep, cfg)\n    X, y = F[feats], F.rul_capped\n    first, last = cfg[\"anomaly_train_cycles\"]\n    healthy = F.cycle.between(first, last)  # early life: no failure labels used\n    iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(F.loc[healthy, acols])\n    return {\n        \"kind\": kind, \"subset\": cfg[\"subset\"], \"cfg\": dict(cfg), \"prep\": prep,\n        \"features\": feats, \"anomaly_features\": acols,\n        \"rul_model\": make_rul_model(kind).fit(X, y),\n        \"q_lo\": make_quantile(0.1).fit(X, y), \"q_hi\": make_quantile(0.9).fit(X, y),\n        \"interval_correction\": float(interval_correction),\n        \"classifier\": make_classifier().fit(X, F.fail), \"clf_threshold\": float(clf_threshold),\n        \"iso\": iso,\n        \"anomaly_threshold\": float(np.quantile(-iso.score_samples(F.loc[healthy, acols]), cfg[\"anomaly_quantile\"])),\n        \"units\": sorted(int(u) for u in df.unit.unique()), \"n_rows\": int(len(df)),\n        \"train_seconds\": round(time.time() - t0, 1), \"created_at\": now_iso(),\n    }\n\n\ndef predict_bundle(b, df):\n    cfg, cap = b[\"cfg\"], b[\"cfg\"][\"rul_cap\"]\n    F = featurize(df, b[\"prep\"], cfg)\n    X = F[b[\"features\"]]\n    out = F[[c for c in META if c in F]].copy()\n    pred = np.clip(b[\"rul_model\"].predict(X), 0, cap)\n    q = b.get(\"interval_correction\", 0.0)\n    out[\"rul_pred\"] = pred\n    out[\"rul_lo\"] = np.minimum(np.clip(b[\"q_lo\"].predict(X) - q, 0, cap), pred)\n    out[\"rul_hi\"] = np.maximum(np.clip(b[\"q_hi\"].predict(X) + q, 0, cap), pred)\n    out[\"p_fail\"] = b[\"classifier\"].predict_proba(X)[:, 1]\n    out[\"fail_alert\"] = out.p_fail >= b[\"clf_threshold\"]\n    out[\"anomaly_score\"] = -b[\"iso\"].score_samples(F[b[\"anomaly_features\"]])\n    out[\"anomaly\"] = out.anomaly_score > b[\"anomaly_threshold\"]\n    k, w = cfg[\"anomaly_persist\"], cfg[\"roll_window\"]\n    g = out.groupby(\"unit\")\n    out[\"anomaly_persistent\"] = g.anomaly.transform(lambda a: a.astype(int).rolling(k, min_periods=k).min().fillna(0)).astype(bool)\n    out[\"recent_imputed\"] = g.imputed.transform(lambda a: a.astype(int).rolling(w, min_periods=1).max()).astype(bool)\n    out[\"recent_unseen\"] = g.unseen_condition.transform(lambda a: a.astype(int).rolling(w, min_periods=1).max()).astype(bool)\n    out[\"cycles_observed\"] = g.cumcount() + 1\n    return out\n\n\ndef evaluate_bundle(b, val_df, test_df):\n    # val_df=None skips validation metrics, for a model that has trained on the validation engines\n    cfg, cap = b[\"cfg\"], b[\"cfg\"][\"rul_cap\"]\n    Pt = predict_bundle(b, test_df)\n    last = latest(Pt)\n    m = prefix(\"test\", rul_metrics(last.rul, last.rul_pred, cap))\n    m[\"test_interval_coverage\"] = interval_coverage(last)\n    m.update(coverage_by_band(last, \"test\"))\n    Pv = None\n    if val_df is not None:\n        Pv = predict_bundle(b, val_df)\n        tl = test_like(Pv, cfg)\n        m.update(prefix(\"val\", rul_metrics(tl.rul, tl.rul_pred, cap)))\n        m.update(prefix(\"val_all\", rul_metrics(Pv.rul, Pv.rul_pred, cap)))\n        m[\"val_interval_coverage\"] = interval_coverage(tl)\n        m.update(coverage_by_band(tl, \"val\"))\n        m.update(prefix(\"val_clf\", clf_metrics(Pv.fail, Pv.p_fail, b[\"clf_threshold\"])))\n        m.update(prefix(\"val_anom\", anomaly_metrics(Pv, cfg)))\n    return m, Pv, Pt\n\n\ndef show_metrics(m, keys=(\"val_rmse_capped\", \"val_rmse\", \"val_nasa_capped_mean\", \"test_rmse_capped\", \"test_rmse\",\n                          \"test_nasa_score\", \"test_mae\", \"test_bias\",\n                          \"test_late_pct\", \"val_interval_coverage\", \"test_interval_coverage\", \"val_clf_pr_auc\", \"val_clf_recall\",\n                          \"val_clf_precision\", \"val_anom_false_alarm_rate_healthy\", \"val_anom_median_rul_at_first_alarm\")):\n    return pd.Series({k: round(m[k], 3) for k in keys if k in m})",
"# ---- ablation: change one design choice at a time, same 5-fold CV on the non-validation training engines ----\n# Base model is HistGradientBoosting for every variant, so only the design choice changes; model choice is its own block.\n# Metrics on held-out 'test-like' rows (>= 31 cycles observed, true RUL <= 200); no test data is used.\ndev = train_df[train_df.unit.isin(SPLITS[\"initial\"] + SPLITS[\"new\"])].reset_index(drop=True)\nfolds = list(GroupKFold(n_splits=CFG[\"cv_folds\"]).split(dev, groups=dev.unit))\nEVAL_CAP = 125  # every variant is scored against the same truth, whatever cap it trained on\n\n\ndef drop_suffix(suffix):\n    return lambda cols, sensors: [c for c in cols if not c.endswith(suffix) or c == \"cycle\"]\n\n\nVARIANTS = {  # name: (cfg overrides -> new preprocessing, column selector, training cap, model)\n    \"baseline (current design)\": ({}, None, 125, \"hgb\"),\n    \"raw sensors only (no rolling features)\": ({}, lambda cols, sensors: [\"cycle\"] + sensors, 125, \"hgb\"),\n    \"no trend (slope) features\": ({}, lambda cols, sensors: [c for c in cols if \"_slope\" not in c], 125, \"hgb\"),\n    \"no drift-from-own-baseline\": ({}, drop_suffix(\"_drift\"), 125, \"hgb\"),\n    \"no cycle count\": ({}, lambda cols, sensors: [c for c in cols if c != \"cycle\"], 125, \"hgb\"),\n    \"shorter windows (5 / 10 / 5)\": ({\"roll_window\": 5, \"slope_window\": 10, \"ewm_span\": 5}, None, 125, \"hgb\"),\n    \"longer windows (30 / 50 / 30)\": ({\"roll_window\": 30, \"slope_window\": 50, \"ewm_span\": 30}, None, 125, \"hgb\"),\n    \"RUL cap 110\": ({}, None, 110, \"hgb\"),\n    \"RUL cap 140\": ({}, None, 140, \"hgb\"),\n    \"no RUL cap\": ({}, None, 10_000, \"hgb\"),\n    \"model: Ridge (linear)\": ({}, None, 125, \"ridge\"),\n}\nif CFG[\"condition_norm\"]:\n    VARIANTS[\"no per-condition normalisation\"] = ({\"condition_norm\": False}, None, 125, \"hgb\")\nif xgb is not None:\n    VARIANTS[\"model: XGBoost\"] = ({}, None, 125, \"xgb\")\n    VARIANTS[\"model: HistGB + XGBoost average\"] = ({}, None, 125, \"ens\")\n\n\ndef fit_predict(kind, Xtr, ytr, Xva, cap):\n    if kind == \"ens\":\n        p = (make_rul_model(\"hgb\").fit(Xtr, ytr).predict(Xva) + make_rul_model(\"xgb\").fit(Xtr, ytr).predict(Xva)) / 2\n    else:\n        p = make_rul_model(kind).fit(Xtr, ytr).predict(Xva)\n    return np.clip(p, 0, cap)\n\n\ndef scores(y, p):\n    d_cap = p - np.minimum(y, EVAL_CAP)\n    near = y <= 60\n    return {\"rmse_capped\": float(np.sqrt(np.mean(d_cap ** 2))), \"rmse_true\": float(np.sqrt(np.mean((p - y) ** 2))),\n            \"rmse_true_rul_le_60\": float(np.sqrt(np.mean((p[near] - y[near]) ** 2))) if near.any() else float(\"nan\"),\n            \"nasa_capped_mean\": float(nasa_score(np.minimum(y, EVAL_CAP), p).mean())}\n\n\ngroups = {}\nfor name, (over, sel, cap_, kind) in VARIANTS.items():\n    groups.setdefault(json.dumps(over, sort_keys=True), []).append(name)\n\noof = {name: [] for name in VARIANTS}\ncalib = []  # baseline quantile models per fold, for the calibration ablation\nt0 = time.time()\nfor key, names in groups.items():\n    cfg_v = {**CFG, **json.loads(key)}\n    for k, (tr_idx, va_idx) in enumerate(folds):\n        tr, va = dev.iloc[tr_idx], dev.iloc[va_idx]\n        prep = fit_prep(tr, cfg_v)\n        Ftr, Fva = featurize(tr, prep, cfg_v), featurize(va, prep, cfg_v)\n        rows = (Fva.cycle >= CFG[\"eval_min_cycles\"]) & (Fva.rul <= CFG[\"eval_max_rul\"])\n        cols_all = feature_cols(prep, cfg_v)\n        for name in names:\n            _, sel, cap_, kind = VARIANTS[name]\n            cols = sel(cols_all, prep[\"sensors\"]) if sel else cols_all\n            p = fit_predict(kind, Ftr[cols], np.minimum(Ftr.rul, cap_), Fva.loc[rows, cols], cap_)\n            oof[name].append(pd.DataFrame({\"fold\": k, \"rul\": Fva.loc[rows, \"rul\"].to_numpy(), \"pred\": p}))\n            if name == \"baseline (current design)\":\n                y = np.minimum(Ftr.rul, 125)\n                calib.append(pd.DataFrame({\"fold\": k, \"rul\": Fva.loc[rows, \"rul\"].to_numpy(),\n                                           \"lo\": make_quantile(0.1).fit(Ftr[cols], y).predict(Fva.loc[rows, cols]),\n                                           \"hi\": make_quantile(0.9).fit(Ftr[cols], y).predict(Fva.loc[rows, cols]),\n                                           \"pred\": p}))\n        print(f\"[{SUBSET}] {', '.join(names)[:90]}... fold {k} ({time.time() - t0:.0f}s)\")\n\nbase = pd.concat(oof[\"baseline (current design)\"])\nbase_fold = base.groupby(\"fold\").apply(lambda d: scores(d.rul.to_numpy(), d.pred.to_numpy())[\"rmse_capped\"])\ntable = []\nfor name in VARIANTS:\n    o = pd.concat(oof[name])\n    per_fold = o.groupby(\"fold\").apply(lambda d: scores(d.rul.to_numpy(), d.pred.to_numpy())[\"rmse_capped\"])\n    delta = per_fold - base_fold\n    table.append({\"variant\": name, **scores(o.rul.to_numpy(), o.pred.to_numpy()),\n                  \"fold_std_rmse_capped\": float(per_fold.std()), \"delta_vs_baseline\": float(delta.mean()),\n                  \"folds_better_than_baseline\": f\"{int((delta < 0).sum())}/{len(delta)}\"})\nablation = pd.DataFrame(table).set_index(\"variant\").round(3)\n\n# calibration ablation, cross-fitted: each fold's margin comes from the other folds\nc = pd.concat(calib)\nc[\"y\"] = np.minimum(c.rul, 125)\ncovered = lambda d, q: ((d.y >= np.minimum(np.clip(d.lo - q, 0, 125), d.pred)) & (d.y <= np.maximum(np.clip(d.hi + q, 0, 125), d.pred)))\nraw_cov, cal_cov, near_raw, near_cal = [], [], [], []\nfor k in sorted(c.fold.unique()):\n    others, this = c[(c.fold != k) & (c.rul <= 125)], c[c.fold == k]\n    q = conformal_correction(others.y, others.lo, others.hi, CFG[\"interval_coverage\"])\n    raw_cov.append(covered(this[this.rul <= 125], 0.0).mean()); cal_cov.append(covered(this[this.rul <= 125], q).mean())\n    near = this[this.rul <= 30]\n    near_raw.append(covered(near, 0.0).mean()); near_cal.append(covered(near, q).mean())\ncalibration = pd.Series({\"coverage_raw_quantiles\": np.mean(raw_cov), \"coverage_calibrated\": np.mean(cal_cov),\n                         \"coverage_raw_rul_le_30\": np.mean(near_raw), \"coverage_calibrated_rul_le_30\": np.mean(near_cal),\n                         \"target\": CFG[\"interval_coverage\"]}).round(3)\n\ndisplay(ablation)\ndisplay(calibration.to_frame(\"interval calibration (cross-fitted)\"))\nABLATION_OUT = OUT / \"ablation\"\nABLATION_OUT.mkdir(parents=True, exist_ok=True)\nablation.to_csv(ABLATION_OUT / \"ablation.csv\")\nsave_json({\"subset\": SUBSET, \"engines\": int(dev.unit.nunique()), \"folds\": CFG[\"cv_folds\"], \"base_model\": \"hgb\",\n           \"eval_cap\": EVAL_CAP, \"variants\": ablation.reset_index().to_dict(orient=\"records\"),\n           \"calibration\": calibration.to_dict()}, ABLATION_OUT / \"ablation.json\")\nABLATION_RESULT = {\"table\": ablation, \"calibration\": calibration}"
]''')
print(len(ABLATION_CELLS), 'cells loaded')

In [ ]:
import gc, shutil, time, traceback
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
WORK = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path.cwd() / "outputs"
shutil.rmtree(WORK / "artifacts", ignore_errors=True)

ablations = {}
for subset in SUBSETS:
    print(f"\n{'=' * 34} {subset} {'=' * 34}")
    ns = {"__name__": "__main__", "display": display}
    t0 = time.time()
    try:
        for i, src in enumerate(ABLATION_CELLS):
            exec(compile(src.replace('SUBSET = "FD001"', f'SUBSET = "{subset}"'), f"<{subset} cell {i}>", "exec"), ns)
        ablations[subset] = ns["ABLATION_RESULT"]
    except Exception:
        traceback.print_exc()
    finally:
        plt.close("all")
        del ns
        gc.collect()
    print(f"{subset}: {'ok' if subset in ablations else 'FAILED'} in {(time.time() - t0) / 60:.1f} min")

## Cross-subset summary

`delta_vs_baseline` is the change in capped RMSE (cycles) against the current design, averaged over the 5 paired folds: positive = that change makes things worse (so the design choice it removes is worth that much), negative = it would be an improvement.

In [ ]:
if ablations:
    order = list(dict.fromkeys(v for r in ablations.values() for v in r["table"].index))  # keep the variant grouping
    delta = pd.DataFrame({s: r["table"]["delta_vs_baseline"] for s, r in ablations.items()}).reindex(order)
    rmse = pd.DataFrame({s: r["table"]["rmse_capped"] for s, r in ablations.items()}).reindex(order)
    wins = pd.DataFrame({s: r["table"]["folds_better_than_baseline"] for s, r in ablations.items()}).reindex(order)
    calib = pd.DataFrame({s: r["calibration"] for s, r in ablations.items()})
    print("capped RMSE (cycles), CV on training engines")
    display(rmse)
    print("change vs current design (cycles; + = worse)")
    display(delta.round(2))
    print("folds where the variant beat the current design")
    display(wins)
    print("interval calibration (cross-fitted)")
    display(calib)
    out = WORK / "artifacts"
    out.mkdir(parents=True, exist_ok=True)
    rmse.to_csv(out / "ablation_rmse_capped.csv")
    delta.to_csv(out / "ablation_delta_vs_baseline.csv")
    wins.to_csv(out / "ablation_folds_better.csv")
    calib.to_csv(out / "ablation_calibration.csv")
    zip_path = shutil.make_archive(str(WORK / "ablation_outputs"), "zip", root_dir=out)
    print("outputs:", zip_path)